# 回归 CE 云端训练笔记本

本笔记本用于对回归挑战代码库执行数据检查、诊断、模型训练、可视化和结果打包。

In [ ]:
!git clone https://github.com/cotwint/regression_ce_research_code.git

Cloning into 'regression_ce_research_code'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 30 (delta 1), reused 30 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 698.57 KiB | 3.66 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
%cd regression_ce_research_code

/content/regression_ce_research_code


In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path
import os, sys, subprocess, textwrap, zipfile, json
import pandas as pd
import numpy as np

SEED = 114514
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
RUN_DIR = ROOT / 'outputs' / 'cloud_run'
DATA_DIR.mkdir(exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('DATA_DIR =', DATA_DIR)
print('RUN_DIR =', RUN_DIR)

ROOT = /content/regression_ce_research_code
DATA_DIR = /content/regression_ce_research_code/data
RUN_DIR = /content/regression_ce_research_code/outputs/cloud_run


## 数据导入

将 `train.csv`、`test.csv` 和 `sample_submission.csv` 放到 `data/` 下。在 Colab 中，可使用下一单元上传 zip 或单个 CSV 文件，或挂载云盘并将文件复制到 `DATA_DIR`。

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = DATA_DIR / name
        target.write_bytes(content)
        if target.suffix.lower() == '.zip':
            with zipfile.ZipFile(target) as zf:
                zf.extractall(DATA_DIR)
    print('Uploaded files:', sorted(p.name for p in DATA_DIR.iterdir()))
except Exception as exc:
    print('Upload helper skipped:', exc)
    print('Current data files:', sorted(p.name for p in DATA_DIR.iterdir()) if DATA_DIR.exists() else [])

KeyboardInterrupt: 

In [21]:
def check_required_files(data_dir: Path):
    required = ['train.csv', 'test.csv', 'sample_submission.csv']
    missing = [name for name in required if not (data_dir / name).exists()]
    print('Data directory:', data_dir)
    print('Available:', sorted(p.name for p in data_dir.iterdir()))
    if missing:
        raise FileNotFoundError('Missing files: ' + ', '.join(missing))
    header_train = pd.read_csv(data_dir / 'train.csv', nrows=2)
    header_test = pd.read_csv(data_dir / 'test.csv', nrows=2)
    print('Train preview shape:', header_train.shape)
    print('Test preview shape:', header_test.shape)
    print('First columns:', header_train.columns[:8].tolist())

check_required_files(DATA_DIR)

Data directory: /content/regression_ce_research_code/data
Available: ['Desktop.zip', 'sample_submission.csv', 'test.csv', 'train.csv']
Train preview shape: (2, 4993)
Test preview shape: (2, 4992)
First columns: ['ID', 'target', '48df886f9', '0deb4b6a8', '34b15f335', 'a8cb14b00', '2f0771a37', '30347e683']


## 带错误分析的命令运行器

In [ ]:
def run_cmd(cmd, cwd=ROOT):
    print('Running:', ' '.join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout[-6000:])
    if proc.returncode != 0:
        print('STDERR tail:')
        print(proc.stderr[-6000:])
        msg = str(proc.stderr)
        if 'No module named' in msg or 'ModuleNotFoundError' in msg:
            print('Likely fix: rerun the pip install cell, or install requirements-extra.txt for optional models.')
        if 'MemoryError' in msg or 'CUDA out of memory' in msg:
            print('Likely fix: reduce model list, lower n_splits, remove --save-models, or run on a larger runtime.')
        if 'FileNotFoundError' in msg:
            print('Likely fix: confirm train.csv/test.csv/sample_submission.csv are under DATA_DIR.')
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)
    if proc.stderr:
        print('STDERR tail:')
        print(proc.stderr[-3000:])
    return proc

## 诊断

In [ ]:
run_cmd([
    sys.executable, '-m', 'src.diagnostics',
    '--data-dir', str(DATA_DIR),
    '--train-file', 'train.csv',
    '--test-file', 'test.csv',
    '--out-dir', str(ROOT / 'outputs' / 'diagnostics')
])

Running: /usr/bin/python3 -m src.diagnostics --data-dir /content/regression_ce_research_code/data --train-file train.csv --test-file test.csv --out-dir /content/regression_ce_research_code/outputs/diagnostics
STDERR tail:
defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return 

CompletedProcess(args=['/usr/bin/python3', '-m', 'src.diagnostics', '--data-dir', '/content/regression_ce_research_code/data', '--train-file', 'train.csv', '--test-file', 'test.csv', '--out-dir', '/content/regression_ce_research_code/outputs/diagnostics'], returncode=0, stdout='', stderr='2026-06-10 03:30:41,693 | INFO | reading train data from /content/regression_ce_research_code/data/train.csv\n2026-06-10 03:30:46,181 | INFO | reading test data from /content/regression_ce_research_code/data/test.csv\n2026-06-10 03:31:24,780 | INFO | train shape: (4459, 4993)\n2026-06-10 03:31:24,790 | INFO | test shape: (49342, 4992)\n/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.\n  return spearmanr(a, b)[0]\n/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.\n  return spearman

## 完整训练

默认列表会训练强健的经典模型和提升树基线。可添加 `extratrees` 以获得更慢的树袋基线，或安装额外依赖并添加 `tabpfn` 来使用可选的表格基础模型包装器。

In [22]:
MODELS = ['mean_log', 'ridge_svd', 'elasticnet_svd', 'lightgbm', 'xgboost', 'catboost']
run_cmd([
    sys.executable, '-m', 'src.train',
    '--data-dir', str(DATA_DIR),
    '--train-file', 'train.csv',
    '--test-file', 'test.csv',
    '--sample-submission', 'sample_submission.csv',
    '--out-dir', str(RUN_DIR),
    '--models', *MODELS,
    '--n-splits', '5',
    '--seed', str(SEED),
    '--svd-components', '128',
    '--pca-dims', '16', '32', '64', '128', '256',
    '--save-models'
])

Running: /usr/bin/python3 -m src.train --data-dir /content/regression_ce_research_code/data --train-file train.csv --test-file test.csv --sample-submission sample_submission.csv --out-dir /content/regression_ce_research_code/outputs/cloud_run --models mean_log ridge_svd elasticnet_svd lightgbm xgboost catboost --n-splits 5 --seed 114514 --svd-components 128 --pca-dims 16 32 64 128 256 --save-models


KeyboardInterrupt: 

## 可视化交叉验证指标与模型影响

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(context='notebook')
BASE_COLOR = sns.color_palette()[0]

metrics = pd.read_csv(RUN_DIR / 'metrics_cv.csv')
summary = pd.read_csv(RUN_DIR / 'metrics_summary.csv')
display(summary.sort_values(['metric', 'mean']).head(30))

valid = metrics[metrics['split'] == 'valid'].copy()
plt.figure(figsize=(10, 5))
sns.barplot(data=valid, x='model', y='rmsle', hue='model', legend=False)
plt.xticks(rotation=35, ha='right')
plt.title('Validation RMSLE by model')
plt.show()

if (RUN_DIR / 'pca_sweep_metrics.csv').exists():
    pca = pd.read_csv(RUN_DIR / 'pca_sweep_metrics.csv')
    plt.figure(figsize=(8, 5))
    sns.lineplot(data=pca[pca['split'] == 'valid'], x='n_components', y='rmsle', marker='o', color=BASE_COLOR)
    plt.title('PCA/SVD dimension sweep')
    plt.show()

In [ ]:
oof = pd.read_csv(RUN_DIR / 'oof_predictions.csv')
pred_cols = [c for c in oof.columns if c.startswith('pred_')]
best = summary[(summary['split'] == 'valid') & (summary['metric'] == 'rmsle')].sort_values('mean').iloc[0]['model']
best_col = f'pred_{best}' if f'pred_{best}' in pred_cols else pred_cols[-1]
sample = oof.sample(min(3000, len(oof)), random_state=SEED)
plt.figure(figsize=(6, 6))
sns.scatterplot(data=sample, x='target', y=best_col, s=12, alpha=0.5, color=BASE_COLOR)
plt.title(f'OOF actual vs predicted: {best_col}')
plt.show()

plt.figure(figsize=(9, 5))
sns.histplot(sample[best_col] - sample['target'], bins=80, kde=True, color=BASE_COLOR)
plt.title(f'Residuals: {best_col}')
plt.show()

## 打包运行输出

In [ ]:
RESULT_ZIP = RUN_DIR / 'result_pack.zip'
run_cmd([
    sys.executable, '-m', 'src.package_results',
    '--run-dir', str(RUN_DIR),
    '--zip-path', str(RESULT_ZIP)
])
print('Result package:', RESULT_ZIP)
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except Exception as exc:
    print('Download helper skipped:', exc)